In [ ]:
from datetime import UTC, datetime

import geopandas as gpd


In [ ]:
SIT_AUTOR_DOSSIER_GDB_PATH = r"C:\Users\denis.iglesias\Downloads\SIT_AUTOR_DOSSIER.gdb"
CAD_BATIMENT_HORSOL_GDB_PATH = (
    r"C:\Users\denis.iglesias\Downloads\CAD_BATIMENT_HORSOL.gdb"
)

In [ ]:
gdf_autor = gpd.read_file(SIT_AUTOR_DOSSIER_GDB_PATH, layer="SIT_AUTOR_DOSSIER")
gdf_autor = gdf_autor[gdf_autor["DATE_DEPOT"] >= datetime(2000, 1, 1, tzinfo=UTC)]

In [ ]:
gdf_autor.head(2)

,ID_DOSSIER,TYPE_DOSSIER,TYPE_OPERATION,NOM_DOSSIER,NO_DOSSIER,NO_COMPLEMENTAIRE,STATUT,DATE_DEPOT,DESCRIPTION,OPERATION,ORIGINE,DATE_MAJ_2,LIEN_SAD,geometry
66585,APA 27165/1,APA,REN,Autorisation procédure accélérée,27165,1.0,TERMINE,2006-09-29 00:00:00+00:00,création de 2 fenêtres sur cour et agrandisse...,Travaux d'entretien et de rénovation intérieur...,SIT,2023-11-28 00:00:00+00:00,https://app2.ge.ch/sadconsult/dossier/APA/27165/1,POINT (2498876.84 1121321.56)
66586,DD 100079/2,DD,NOU,Demande définitive,100079,2.0,ACCEPTE,2009-03-24 00:00:00+00:00,"""GENEVE PLAGE""(transformation et surélévation ...",Construction nouvelle importante ou extension ...,SIT,2023-11-28 00:00:00+00:00,https://app2.ge.ch/sadconsult/dossier/DD/100079/2,POINT (2502357.69 1118833.96)


In [ ]:
gdf_batiment_horsol = gpd.read_file(CAD_BATIMENT_HORSOL_GDB_PATH)

In [ ]:
gdf_batiment_horsol.drop(
    columns=[
        "NO_COMM",
        "NO_BATIMENT",
        "IDENT",
        "NOMBAT",
        "MUTNUM",
        "PROVENANCE",
        "NO_AUTOR",
        "EGRID_LISTE",
        "EGRID_CENTROIDE",
    ],
    inplace=True,
)

In [ ]:
gdf_batiment_horsol.head(3)

,COMMUNE,EGID,DATEDT,DESTINATION,NOMENCLATURE,NOMEN_CLASSE,EPOQUE_CONSTRUCTION,ANNEE_CONSTRUCTION,ANNEE_TRANSFORNATION,NIVEAUX_HORSOL,NIVEAUX_SSOL,HAUTEUR,SURFACE,SHAPE_Length,SHAPE_Area,geometry
0,Veyrier,1031173,2016-09-13 15:06:44+00:00,Habitation un logement,1.1.1,Habitation,Période de 1981 à 1985,NaN,NaN,3.0,NaN,4.48,91,46.294559,90.705299,"MULTIPOLYGON (((2500366.216 1114228.42, 250037..."
1,Veyrier,295073184,2006-01-20 17:30:57+00:00,Autre bât. 20m2 et plus,5.1,Autre bâtiment,Période de 1971 à 1980,NaN,NaN,NaN,NaN,5.84,91,42.235823,90.683050,"MULTIPOLYGON (((2502942.38 1113676.281, 250293..."
2,Veyrier,2749270,1998-12-04 00:00:00+00:00,Habitation un logement,1.1.1,Habitation,Période de 1996 à 2000,NaN,NaN,3.0,NaN,7.99,109,42.157512,109.286539,"MULTIPOLYGON (((2502777.214 1114203.843, 25027..."


In [ ]:
# Buffer of 1m on points (CRS must be metric — EPSG:2056 is fine)
gdf_autor_buffered = gdf_autor.copy()
gdf_autor_buffered["geometry"] = gdf_autor.buffer(1)

# Spatial join — keep only points inside polygons
gdf_joined = gpd.sjoin(
    gdf_autor_buffered, gdf_batiment_horsol, how="inner", predicate="intersects"
)

In [ ]:
gdf_joined.columns

Index(['ID_DOSSIER', 'TYPE_DOSSIER', 'TYPE_OPERATION', 'NOM_DOSSIER',
       'NO_DOSSIER', 'NO_COMPLEMENTAIRE', 'STATUT', 'DATE_DEPOT',
       'DESCRIPTION', 'OPERATION', 'ORIGINE', 'DATE_MAJ_2', 'LIEN_SAD',
       'geometry', 'index_right', 'COMMUNE', 'EGID', 'DATEDT', 'DESTINATION',
       'NOMENCLATURE', 'NOMEN_CLASSE', 'EPOQUE_CONSTRUCTION',
       'ANNEE_CONSTRUCTION', 'ANNEE_TRANSFORNATION', 'NIVEAUX_HORSOL',
       'NIVEAUX_SSOL', 'HAUTEUR', 'SURFACE', 'SHAPE_Length', 'SHAPE_Area'],
      dtype='str')